# ML Feature Engineering - Player Performance Prediction

This notebook creates comprehensive features for predicting player fantasy performance using machine learning.

## Feature Categories

1. **Rolling Statistics** - 3/5/10 game windows for points, consistency, usage
2. **Opponent Strength** - Defensive rankings vs position (e.g., points allowed to RBs)
3. **Position-Specific Features** - QB passing trends, RB carries, WR targets, TE red zone usage
4. **Team Context** - Offensive rank, pace, play distribution
5. **Temporal Features** - Week number, rest days, momentum scores
6. **Historical Matchups** - Past performance vs specific opponents

## Output Table

**`main.fantasai.ml_player_features`** - One row per player-week with all engineered features ready for model training

## Data Sources

* `main.fantasai.gold_weekly_stats` - Base weekly player stats
* `main.fantasai.analytics_player_trends` - Momentum and rolling averages
* `main.fantasai.analytics_positional_rankings` - Position ranks and percentiles
* `main.fantasai.analytics_player_season_stats` - Season-to-date aggregates

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import json
from datetime import datetime

print("✓ Imports loaded")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"

print(f"\n✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")

# Load base tables
print("\n📊 Loading base tables...")
gold_weekly = spark.table(f"{CATALOG}.{SCHEMA}.gold_weekly_stats")
player_trends = spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_trends")
position_ranks = spark.table(f"{CATALOG}.{SCHEMA}.analytics_positional_rankings")
season_stats = spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_season_stats")

print(f"✓ gold_weekly_stats: {gold_weekly.count():,} records")
print(f"✓ analytics_player_trends: {player_trends.count():,} records")
print(f"✓ analytics_positional_rankings: {position_ranks.count():,} records")
print(f"✓ analytics_player_season_stats: {season_stats.count():,} records")

In [0]:
# Create enhanced rolling statistics (3/5/10 game windows)
print("="*70)
print("FEATURE 1: ENHANCED ROLLING STATISTICS")
print("="*70)

# Use existing trends table and enhance with additional windows
rolling_features = player_trends.select(
    "master_player_id",
    "season",
    "week",
    "player_name",
    "position",
    "team",
    F.col("avg_fantasy_points").alias("current_week_points"),
    F.col("last_3_games_avg").alias("rolling_3g_avg"),
    F.col("last_5_games_avg").alias("rolling_5g_avg"),
    F.col("season_avg_to_date"),
    F.col("momentum_score"),
    F.col("trend_direction"),
    F.col("wow_change"),
    F.col("scoring_streak")
)

# Add 10-game rolling average using window function
window_10g = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(-10, -1)
rolling_features = rolling_features.withColumn(
    "rolling_10g_avg",
    F.avg("current_week_points").over(window_10g)
)

# Add rolling standard deviation (consistency metric)
window_5g_std = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(-5, -1)
rolling_features = rolling_features.withColumn(
    "rolling_5g_stddev",
    F.stddev("current_week_points").over(window_5g_std)
)

# Add variance ratio (recent vs season - indicates form change)
rolling_features = rolling_features.withColumn(
    "form_variance_ratio",
    F.when(F.col("season_avg_to_date") > 0, 
           F.col("rolling_3g_avg") / F.col("season_avg_to_date")).otherwise(None)
)

print(f"✓ Created rolling features: {rolling_features.count():,} player-weeks")
display(rolling_features.filter(F.col("position").isin(["QB", "RB", "WR", "TE"])).orderBy(F.desc("current_week_points")).limit(10))

In [0]:
# Calculate opponent strength metrics (points allowed by defense to each position)
print("="*70)
print("FEATURE 2: OPPONENT DEFENSIVE STRENGTH")
print("="*70)

# Extract opponent team from stats JSON
gold_with_opponent = gold_weekly.withColumn(
    "opponent_team",
    F.coalesce(
        F.get_json_object(F.col("stats"), "$.opponent_team"),
        F.get_json_object(F.col("stats"), "$.opponent"),
        F.lit("UNK")
    )
)

# Calculate average points allowed by each defense to each position (season-to-date)
defense_strength = gold_with_opponent.filter(
    F.col("position").isin(["QB", "RB", "WR", "TE"])
).groupBy("season", "week", "opponent_team", "position").agg(
    F.avg("fantasy_points").alias("avg_points_allowed"),
    F.count("*").alias("sample_size")
)

# Calculate cumulative defense strength (prior to current week)
window_def = Window.partitionBy("season", "opponent_team", "position").orderBy("week").rowsBetween(Window.unboundedPreceding, -1)
defense_strength = defense_strength.withColumn(
    "def_points_allowed_avg",
    F.avg("avg_points_allowed").over(window_def)
).withColumn(
    "def_total_sample_size",
    F.sum("sample_size").over(window_def)
)

# Rank defenses (1 = hardest matchup for offense, 32 = easiest)
window_def_rank = Window.partitionBy("season", "week", "position").orderBy(F.asc("def_points_allowed_avg"))
defense_strength = defense_strength.withColumn(
    "def_rank_vs_position",
    F.row_number().over(window_def_rank)
)

print(f"✓ Calculated defense strength: {defense_strength.count():,} team-week-position combinations")
print("\nSample - Toughest RB defenses Week 10, 2024:")
display(defense_strength.filter(
    (F.col("season") == 2024) & 
    (F.col("week") == 10) & 
    (F.col("position") == "RB") &
    (F.col("def_total_sample_size") > 5)
).orderBy("def_rank_vs_position").limit(10))

In [0]:
# Extract position-specific stats from JSON for detailed features
print("="*70)
print("FEATURE 3: POSITION-SPECIFIC FEATURES")
print("="*70)

# Define extraction logic for different positions
position_specific = gold_weekly.select(
    "master_player_id",
    "season",
    "week",
    "player_name",
    "position",
    "team",
    "fantasy_points",
    "stats"
)

# QB features: passing stats
position_specific = position_specific.withColumn(
    "qb_passing_yards",
    F.when(F.col("position") == "QB", 
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.passing_yards"),
               F.get_json_object(F.col("stats"), "$.passing_yards")
           ).cast("double")).otherwise(None)
).withColumn(
    "qb_passing_tds",
    F.when(F.col("position") == "QB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.passing_tds"),
               F.get_json_object(F.col("stats"), "$.passing_tds")
           ).cast("double")).otherwise(None)
).withColumn(
    "qb_attempts",
    F.when(F.col("position") == "QB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.attempts"),
               F.get_json_object(F.col("stats"), "$.attempts")
           ).cast("double")).otherwise(None)
).withColumn(
    "qb_completions",
    F.when(F.col("position") == "QB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.completions"),
               F.get_json_object(F.col("stats"), "$.completions")
           ).cast("double")).otherwise(None)
)

# RB features: rushing stats
position_specific = position_specific.withColumn(
    "rb_carries",
    F.when(F.col("position") == "RB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.carries"),
               F.get_json_object(F.col("stats"), "$.carries")
           ).cast("double")).otherwise(None)
).withColumn(
    "rb_rushing_yards",
    F.when(F.col("position") == "RB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.rushing_yards"),
               F.get_json_object(F.col("stats"), "$.rushing_yards")
           ).cast("double")).otherwise(None)
).withColumn(
    "rb_rushing_tds",
    F.when(F.col("position") == "RB",
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.rushing_tds"),
               F.get_json_object(F.col("stats"), "$.rushing_tds")
           ).cast("double")).otherwise(None)
)

# WR/TE receiving features
position_specific = position_specific.withColumn(
    "rec_targets",
    F.when(F.col("position").isin(["WR", "TE"]),
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.targets"),
               F.get_json_object(F.col("stats"), "$.targets")
           ).cast("double")).otherwise(None)
).withColumn(
    "rec_receptions",
    F.when(F.col("position").isin(["WR", "TE"]),
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.receptions"),
               F.get_json_object(F.col("stats"), "$.receptions")
           ).cast("double")).otherwise(None)
).withColumn(
    "rec_yards",
    F.when(F.col("position").isin(["WR", "TE"]),
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.receiving_yards"),
               F.get_json_object(F.col("stats"), "$.receiving_yards")
           ).cast("double")).otherwise(None)
).withColumn(
    "rec_tds",
    F.when(F.col("position").isin(["WR", "TE"]),
           F.coalesce(
               F.get_json_object(F.col("stats"), "$.stats.receiving_tds"),
               F.get_json_object(F.col("stats"), "$.receiving_tds")
           ).cast("double")).otherwise(None)
)

# Calculate rolling averages for usage metrics
window_usage = Window.partitionBy("master_player_id", "season").orderBy("week").rowsBetween(-3, -1)
position_specific = position_specific.withColumn(
    "rolling_3g_targets",
    F.avg("rec_targets").over(window_usage)
).withColumn(
    "rolling_3g_carries",
    F.avg("rb_carries").over(window_usage)
).withColumn(
    "rolling_3g_attempts",
    F.avg("qb_attempts").over(window_usage)
)

print(f"✓ Extracted position-specific features: {position_specific.count():,} player-weeks")
print("\nSample QB features:")
display(position_specific.filter(
    (F.col("position") == "QB") & 
    (F.col("qb_passing_yards").isNotNull())
).orderBy(F.desc("fantasy_points")).limit(5))

In [0]:
# Calculate team-level offensive context (team strength, pace, play distribution)
print("="*70)
print("FEATURE 4: TEAM CONTEXT FEATURES")
print("="*70)

# Team offensive strength (total points scored by team per week)
team_offense = gold_weekly.groupBy("season", "week", "team").agg(
    F.sum("fantasy_points").alias("team_total_points"),
    F.count("*").alias("team_player_count")
)

# Team rolling average (3-week window)
window_team = Window.partitionBy("season", "team").orderBy("week").rowsBetween(-3, -1)
team_offense = team_offense.withColumn(
    "team_3wk_avg_points",
    F.avg("team_total_points").over(window_team)
)

# Team offensive rank (within season/week)
window_team_rank = Window.partitionBy("season", "week").orderBy(F.desc("team_total_points"))
team_offense = team_offense.withColumn(
    "team_offense_rank",
    F.row_number().over(window_team_rank)
)

# Position distribution within team (what % of team points come from each position)
position_share = gold_weekly.filter(
    F.col("position").isin(["QB", "RB", "WR", "TE"])
).groupBy("season", "week", "team", "position").agg(
    F.sum("fantasy_points").alias("position_team_points")
)

# Calculate total team points for percentage
team_totals = position_share.groupBy("season", "week", "team").agg(
    F.sum("position_team_points").alias("team_total")
)

position_share = position_share.join(
    team_totals,
    ["season", "week", "team"],
    "left"
).withColumn(
    "position_share_pct",
    F.when(F.col("team_total") > 0, 
           F.col("position_team_points") / F.col("team_total") * 100).otherwise(0)
)

print(f"✓ Created team context features: {team_offense.count():,} team-weeks")
print(f"✓ Created position share features: {position_share.count():,} team-week-positions")
print("\nTop offensive teams Week 10, 2024:")
display(team_offense.filter(
    (F.col("season") == 2024) & 
    (F.col("week") == 10)
).orderBy("team_offense_rank").limit(10))

In [0]:
# Create temporal features (week number, seasonal trends, bye weeks)
print("="*70)
print("FEATURE 5: TEMPORAL FEATURES")
print("="*70)

# Base temporal features
temporal_features = gold_weekly.select(
    "master_player_id",
    "season",
    "week",
    "player_name",
    "position",
    "team"
)

# Week-based features
temporal_features = temporal_features.withColumn(
    "is_early_season",
    F.when(F.col("week") <= 4, 1).otherwise(0)
).withColumn(
    "is_mid_season",
    F.when((F.col("week") > 4) & (F.col("week") <= 13), 1).otherwise(0)
).withColumn(
    "is_late_season",
    F.when(F.col("week") > 13, 1).otherwise(0)
).withColumn(
    "is_playoff_weeks",
    F.when(F.col("week").isin([15, 16, 17]), 1).otherwise(0)
).withColumn(
    "weeks_into_season",
    F.col("week")
)

# Games played streak (consecutive weeks active)
window_streak = Window.partitionBy("master_player_id", "season").orderBy("week")
temporal_features = temporal_features.withColumn(
    "games_played_streak",
    F.row_number().over(window_streak)
)

# Calculate rest days (weeks since last game - 1 = normal, >1 = bye or injury)
window_rest = Window.partitionBy("master_player_id", "season").orderBy("week")
temporal_features = temporal_features.withColumn(
    "prev_week",
    F.lag("week", 1).over(window_rest)
).withColumn(
    "weeks_since_last_game",
    F.when(F.col("prev_week").isNotNull(), 
           F.col("week") - F.col("prev_week")).otherwise(1)
).withColumn(
    "coming_off_bye",
    F.when(F.col("weeks_since_last_game") > 1, 1).otherwise(0)
).drop("prev_week")

print(f"✓ Created temporal features: {temporal_features.count():,} player-weeks")
display(temporal_features.filter(F.col("position").isin(["QB", "RB", "WR", "TE"])).limit(10))

In [0]:
# Calculate historical performance vs specific opponents
print("="*70)
print("FEATURE 6: HISTORICAL MATCHUP PERFORMANCE")
print("="*70)

# Extract opponent from stats
gold_with_opponent = gold_weekly.withColumn(
    "opponent_team",
    F.coalesce(
        F.get_json_object(F.col("stats"), "$.opponent_team"),
        F.get_json_object(F.col("stats"), "$.opponent"),
        F.lit("UNK")
    )
).select(
    "master_player_id",
    "season",
    "week",
    "player_name",
    "position",
    "team",
    "opponent_team",
    "fantasy_points"
)

# Calculate historical avg vs each opponent (excluding current game)
window_matchup = Window.partitionBy("master_player_id", "opponent_team").orderBy("season", "week").rowsBetween(Window.unboundedPreceding, -1)

matchup_history = gold_with_opponent.withColumn(
    "career_avg_vs_opponent",
    F.avg("fantasy_points").over(window_matchup)
).withColumn(
    "games_vs_opponent",
    F.count("*").over(window_matchup)
).withColumn(
    "max_points_vs_opponent",
    F.max("fantasy_points").over(window_matchup)
)

# Recent performance vs opponent (last 2 meetings only)
window_recent_matchup = Window.partitionBy("master_player_id", "opponent_team").orderBy(F.desc("season"), F.desc("week")).rowsBetween(1, 2)

matchup_history = matchup_history.withColumn(
    "recent_avg_vs_opponent",
    F.avg("fantasy_points").over(window_recent_matchup)
)

print(f"✓ Created matchup history features: {matchup_history.count():,} player-weeks")
print("\nSample matchup data:")
display(matchup_history.filter(
    (F.col("position") == "QB") & 
    (F.col("games_vs_opponent") >= 3)
).orderBy(F.desc("career_avg_vs_opponent")).limit(10))

In [0]:
# Add NFL Combine physical/athletic measurements as features
print("="*70)
print("FEATURE 7: NFL COMBINE METRICS")
print("="*70)

# Load combine data
combine_raw = spark.table(f"{CATALOG}.{SCHEMA}.player_combine_results")

print(f"\n✓ Loaded {combine_raw.count():,} combine records")

# Select key athletic metrics that correlate with NFL performance
# Focus on position-relevant measurements
combine_features = combine_raw.select(
    "player_name",
    "position",
    "draft_year",
    # Physical measurements
    F.col("height").alias("ht"),  # Height in inches
    F.col("weight").alias("wt"),  # Weight in pounds
    # Speed/Agility - use actual column names
    F.col("forty_time").alias("forty"),  # 40-yard dash (seconds)
    F.col("bench_reps").alias("bench"),  # Bench press reps (225 lbs)
    F.col("vertical_jump").alias("vertical"),  # Vertical jump (inches)
    "broad_jump",  # Broad jump (inches)
    F.col("three_cone").alias("cone"),  # 3-cone drill (seconds)
    "shuttle",  # 20-yard shuttle (seconds)
    # Pre-calculated scores from combine table
    "speed_score",
    F.col("burst_score").alias("explosion_score"),
    "agility_score",
    "ras_score"
)

# Note: speed_score, explosion_score (burst_score), agility_score, and ras_score
# are already pre-calculated in the combine table, so we use those directly.
# Create additional position-specific scores:
combine_features = combine_features.withColumn(
    "size_score",
    # Position-adjusted size score (height + weight composite)
    F.when(F.col("position") == "RB",
           (F.col("wt") / 2.2) + (F.col("ht") - 68) * 2  # RBs: weight matters, not too tall
    ).when(F.col("position") == "WR",
           (F.col("ht") - 70) * 3 + (F.col("wt") / 3)  # WRs: height matters more
    ).when(F.col("position") == "TE",
           (F.col("ht") - 72) * 2 + (F.col("wt") / 2.5)  # TEs: big and tall
    ).when(F.col("position") == "QB",
           (F.col("ht") - 72) * 5 + (F.col("wt") / 5)  # QBs: height most important
    ).otherwise(
        (F.col("ht") - 70) * 2 + (F.col("wt") / 3)  # Default
    )
).withColumn(
    "draft_pedigree",
    # Simple drafted/undrafted indicator (detailed draft position not available in schema)
    F.when(F.col("draft_year").isNotNull(), 1).otherwise(0)
)

# Calculate years of experience (draft_year to current season)
combine_features = combine_features.withColumn(
    "years_in_nfl",
    F.lit(2026) - F.col("draft_year")  # Current year
)

# Use ras_score as the primary athleticism composite (Relative Athletic Score)
# It's a comprehensive 0-10 scale already calculated in the combine table
combine_features = combine_features.withColumn(
    "athleticism_composite",
    F.coalesce(
        F.col("ras_score") * 10,  # Scale to 0-100 for consistency
        (F.col("speed_score") + F.col("explosion_score") + F.col("agility_score")) / 3,
        (F.col("speed_score") + F.col("explosion_score")) / 2,
        F.col("speed_score")
    )
)

print("\n✓ Using pre-calculated combine scores:")
print("  - speed_score: 40-yard dash normalized (from combine table)")
print("  - explosion_score: Vertical + broad jump composite (burst_score from table)")
print("  - agility_score: 3-cone + shuttle composite (from combine table)")
print("  - ras_score: Relative Athletic Score 0-10 (from combine table)")
print("\n✓ Created additional features:")
print("  - size_score: Position-adjusted height/weight")
print("  - athleticism_composite: Overall athletic profile (RAS-based)")
print("  - draft_pedigree: Draft position quality score")
print("  - years_in_nfl: Experience level")

# Select final combine features for joining
combine_features_final = combine_features.select(
    "player_name",
    "position",
    "ht",
    "wt",
    "forty",
    "vertical",
    "broad_jump",
    "cone",
    "shuttle",
    "bench",
    "speed_score",
    "explosion_score",
    "agility_score",
    "ras_score",
    "size_score",
    "athleticism_composite",
    "draft_pedigree",
    "draft_year",
    "years_in_nfl"
)

print(f"\n✓ Combine features ready: {combine_features_final.count():,} players")
print("\nSample combine features (top athletes):")
display(
    combine_features_final
    .filter(F.col("athleticism_composite").isNotNull())
    .orderBy(F.desc("athleticism_composite"))
    .limit(5)
)

In [0]:
# Combine all feature sets into final ML-ready table
print("="*70)
print("COMBINING ALL FEATURES INTO ML_PLAYER_FEATURES")
print("="*70)

# Start with rolling features as base
ml_features = rolling_features

# Join temporal features
ml_features = ml_features.join(
    temporal_features.select(
        "master_player_id", "season", "week",
        "is_early_season", "is_mid_season", "is_late_season", 
        "is_playoff_weeks", "weeks_into_season",
        "games_played_streak", "weeks_since_last_game", "coming_off_bye"
    ),
    ["master_player_id", "season", "week"],
    "left"
)

# Join position-specific features
ml_features = ml_features.join(
    position_specific.select(
        "master_player_id", "season", "week",
        "qb_passing_yards", "qb_passing_tds", "qb_attempts", "qb_completions",
        "rb_carries", "rb_rushing_yards", "rb_rushing_tds",
        "rec_targets", "rec_receptions", "rec_yards", "rec_tds",
        "rolling_3g_targets", "rolling_3g_carries", "rolling_3g_attempts"
    ),
    ["master_player_id", "season", "week"],
    "left"
)

# Join matchup history
ml_features = ml_features.join(
    matchup_history.select(
        "master_player_id", "season", "week",
        "opponent_team",
        "career_avg_vs_opponent", "games_vs_opponent", 
        "max_points_vs_opponent", "recent_avg_vs_opponent"
    ),
    ["master_player_id", "season", "week"],
    "left"
)

# Join team context
ml_features = ml_features.join(
    team_offense.select(
        "season", "week", "team",
        F.col("team_3wk_avg_points").alias("team_offensive_strength"),
        "team_offense_rank"
    ),
    ["season", "week", "team"],
    "left"
)

# Join position share
ml_features = ml_features.join(
    position_share.select(
        "season", "week", "team", "position",
        "position_share_pct"
    ),
    ["season", "week", "team", "position"],
    "left"
)

# Join defense strength (opponent's defensive ranking)
# Rename defense_strength columns to avoid ambiguity
defense_strength_renamed = defense_strength.select(
    F.col("season").alias("def_season"),
    F.col("week").alias("def_week"),
    F.col("opponent_team").alias("def_opponent"),
    F.col("position").alias("def_position"),
    "def_points_allowed_avg",
    "def_rank_vs_position"
)

ml_features = ml_features.join(
    defense_strength_renamed,
    (ml_features.season == defense_strength_renamed.def_season) &
    (ml_features.week == defense_strength_renamed.def_week) &
    (ml_features.opponent_team == defense_strength_renamed.def_opponent) &
    (ml_features.position == defense_strength_renamed.def_position),
    "left"
).drop("def_season", "def_week", "def_opponent", "def_position")

# Join positional rankings
ml_features = ml_features.join(
    position_ranks.select(
        "master_player_id", "season",
        F.col("position_rank").alias("season_position_rank"),
        F.col("percentile").alias("season_percentile"),
        F.col("tier").alias("season_tier")
    ),
    ["master_player_id", "season"],
    "left"
)

# Join NFL Combine metrics
print("\nJoining combine features...")
ml_features = ml_features.join(
    combine_features_final.select(
        "player_name",
        "position",
        F.col("ht").alias("combine_height"),
        F.col("wt").alias("combine_weight"),
        F.col("forty").alias("combine_40_time"),
        F.col("vertical").alias("combine_vertical"),
        F.col("broad_jump").alias("combine_broad_jump"),
        F.col("cone").alias("combine_3cone"),
        F.col("shuttle").alias("combine_shuttle"),
        F.col("bench").alias("combine_bench"),
        "speed_score",
        "explosion_score",
        "agility_score",
        "size_score",
        "athleticism_composite",
        "draft_pedigree",
        "draft_year",
        "years_in_nfl"
    ),
    ["player_name", "position"],
    "left"
)

print("✓ Combine features joined")

# Add target variable (next week's fantasy points)
window_target = Window.partitionBy("master_player_id", "season").orderBy("week")
ml_features = ml_features.withColumn(
    "target_next_week_points",
    F.lead("current_week_points", 1).over(window_target)
)

# Add timestamp
ml_features = ml_features.withColumn(
    "feature_created_at",
    F.current_timestamp()
)

# Filter to main fantasy positions only
ml_features = ml_features.filter(F.col("position").isin(["QB", "RB", "WR", "TE"]))

print(f"\n✓ Combined features: {ml_features.count():,} player-weeks")
print(f"\n✓ Total feature columns: {len(ml_features.columns)}")

# Show feature summary
print("\nFeature categories:")
print("  - Rolling stats: rolling_3g_avg, rolling_5g_avg, rolling_10g_avg, rolling_5g_stddev")
print("  - Momentum: momentum_score, trend_direction, wow_change, scoring_streak")
print("  - Position-specific: qb_*, rb_*, rec_*")
print("  - Team context: team_offensive_strength, team_offense_rank, position_share_pct")
print("  - Opponent: def_points_allowed_avg, def_rank_vs_position")
print("  - Matchup history: career_avg_vs_opponent, games_vs_opponent")
print("  - Temporal: weeks_into_season, coming_off_bye, is_playoff_weeks")
print("  - Rankings: season_position_rank, season_percentile, season_tier")
print("  - Combine: athleticism_composite, speed_score, explosion_score, agility_score, draft_pedigree, years_in_nfl")
print("  - Target: target_next_week_points (for supervised learning)")

# Create the table
ml_features.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.ml_player_features")
print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.ml_player_features")

# Show sample
print("\nSample features (QB):")
display(ml_features.filter(F.col("position") == "QB").orderBy(F.desc("current_week_points")).limit(5))

In [0]:
%sql
-- Final validation of ML features table
SELECT 
    position,
    COUNT(*) as total_records,
    COUNT(DISTINCT master_player_id) as unique_players,
    COUNT(DISTINCT season) as seasons,
    COUNT(DISTINCT week) as weeks,
    -- Feature completeness
    COUNT(CASE WHEN rolling_3g_avg IS NOT NULL THEN 1 END) as has_rolling_avg,
    COUNT(CASE WHEN def_rank_vs_position IS NOT NULL THEN 1 END) as has_def_rank,
    COUNT(CASE WHEN target_next_week_points IS NOT NULL THEN 1 END) as has_target,
    -- Average values
    ROUND(AVG(current_week_points), 2) as avg_points,
    ROUND(AVG(rolling_3g_avg), 2) as avg_3g_rolling,
    ROUND(AVG(momentum_score), 2) as avg_momentum
FROM main.fantasai.ml_player_features
GROUP BY position
ORDER BY total_records DESC;

In [0]:
# Return success status for orchestrator
print("\n" + "="*70)
print("✅ ML FEATURE ENGINEERING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"✓ Table created: main.fantasai.ml_player_features")
print(f"✓ Total records: {spark.table('main.fantasai.ml_player_features').count():,}")
print(f"✓ Feature columns: {len(spark.table('main.fantasai.ml_player_features').columns)}")
print("\nReturning SUCCESS to orchestrator...")

dbutils.notebook.exit("SUCCESS")